# 连接文书信息并同步更新7、8、9

从时间数据连接法院和全文，随后执行共享记录去重规则。左表行数必须保持不变，随后再执行明确的删除步骤。

文书连接沿用历史的“原始案号首次出现”规则，以免本次记录去重静默改变原有全文关联。源数据中同案号存在多个法院或不同全文的情况单独输出核查清单；该历史规则不是案件／事故唯一性的证明。本次不自动修复上游来源歧义。

先运行3.4的时间数据合并单元，再运行本文件。输出直接替换，不保留旧版备份。

In [ ]:
# Embedded original record_deduplication.py; CLI entry point omitted.
"""Conservative record/case deduplication and synchronized in-place release.

No accident linkage, no model calls, no source backups. Stage and validate every
output before replacement. Audit row numbers refer to the pre-update files.
"""
from pathlib import Path
import argparse, hashlib, json, re, os, unicodedata
from datetime import datetime
from decimal import Decimal, InvalidOperation
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd
import duckdb

CORE=['case_number','privy','province','city','district','specific_place','vehicle','death','lng','lat','case_dt_hour','court_name','case_text']
TIME=['案号','裁判日期','当事人','province','city','district','specific_place','vehicle','death','lng','lat','case_dt_hour']
SHARED=['case_number','privy','province','city','district','specific_place','vehicle','death','lng','lat','case_dt_hour']
NUMERIC=['vehicle','death','lng','lat']
CIVIL=r'民(?:一|二|三)?初(?:字)?'
CRIMINAL=r'刑(?:一|二)?初(?:字)?'
CHUNK=10000

def normal(s):
    # Preserve punctuation other than bracket width; do NOT erase corrupted '?' case identifiers.
    return re.sub(r'\s+','',unicodedata.normalize('NFKC',str(s)))

def num(s):
    if not s:return ''
    try:
        d=Decimal(s)
        return str(d.normalize()) if d.is_finite() else s
    except InvalidOperation:return s

def sha(values):
    return hashlib.sha256(json.dumps(list(values),ensure_ascii=False,separators=(',',':')).encode('utf-8')).hexdigest()

def first_instance(values):
    v=values.map(normal)
    return v.str.contains(CIVIL,regex=True,na=False)|v.str.contains(CRIMINAL,regex=True,na=False)

def read(path):
    return pd.read_csv(path,dtype=str,keep_default_na=False,chunksize=CHUNK)

class Cursor:
    def __init__(self,frames):self.frames=iter(frames);self.buf=None
    def take(self,n):
        pieces=[]
        while n:
            if self.buf is None or self.buf.empty:
                self.buf=next(self.frames,None)
                if self.buf is None:break
            k=min(n,len(self.buf));pieces.append(self.buf.iloc[:k]);self.buf=self.buf.iloc[k:];n-=k
        return pd.concat(pieces,ignore_index=True) if pieces else pd.DataFrame()

def stat(path):
    s=path.stat();return {'bytes':s.st_size,'mtime_ns':s.st_mtime_ns}

def jsonout(path,value):
    path.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding='utf-8')

class Sink:
    def __init__(self,path,columns):
        self.path=path;self.columns=columns;self.count=0;self.digest=hashlib.sha256()
        self.f=path.open('wb');self._write('\ufeff'+pd.DataFrame(columns=columns).to_csv(index=False,lineterminator='\n'))
    def _write(self,s):
        b=s.encode('utf-8');self.f.write(b);self.digest.update(b)
    def add(self,df):
        if len(df):self._write(df[self.columns].to_csv(index=False,header=False,lineterminator='\n'));self.count+=len(df)
    def close(self):
        self.f.flush();os.fsync(self.f.fileno());self.f.close()
        actual=hashlib.sha256()
        with self.path.open('rb') as f:
            while b:=f.read(8*1024*1024):actual.update(b)
        assert actual.digest()==self.digest.digest(),f'Written-byte checksum failed: {self.path}'
        assert pd.read_csv(self.path,nrows=0).columns.tolist()==self.columns
        return {'rows':self.count,'bytes':self.path.stat().st_size,'sha256':actual.hexdigest()}

def assert_shared(a,t):
    b=t.rename(columns={'案号':'case_number','当事人':'privy'})
    assert len(a)==len(b),'7/8 row counts differ'
    for col in SHARED:
        x=a[col].reset_index(drop=True);y=b[col].reset_index(drop=True)
        if col in NUMERIC:
            nx=pd.to_numeric(x,errors='coerce');ny=pd.to_numeric(y,errors='coerce')
            equal=(x==y)|(nx.notna() & ny.notna() & np.isclose(nx,ny,rtol=0,atol=1e-10))
        else:equal=(x==y)
        assert bool(equal.all()),f'7/8 alignment mismatch in {col}; refusing update'

def synchronize(root,audit_dir=None,commit=True):
    root=Path(root).resolve()
    stamp=datetime.now(ZoneInfo('Asia/Shanghai')).strftime('%Y%m%d_%H%M%S')
    audit=Path(audit_dir).resolve() if audit_dir else root/'检查重复'/('原地更新_'+stamp)
    audit.mkdir(parents=True,exist_ok=False)
    stage=audit/'staging';stage.mkdir()
    def log(s):
        print(s,flush=True)
        with (audit/'run.log').open('a',encoding='utf-8') as f:f.write(s+'\n')
    d7=root/'data/7.匹配时间';d8=root/'data/8.连接数据';d9=root/'data/9.筛选一审'
    parts=[d7/f'time_transport_result_{i}_4.csv' for i in range(1,5)]
    merged=d7/'merged_output.csv';full=d8/'ChinaTransportDatas.csv';first=d9/'ChinaTransportDatas_first_instance.csv'
    inputs=parts+[merged,full,first]
    before={str(p):stat(p) for p in inputs}
    jsonout(audit/'input_manifest.json',before)
    c=duckdb.connect(str(audit/'decisions.duckdb'));c.execute("SET memory_limit='4GB'");c.execute('SET threads=4')
    c.execute("SET temp_directory='"+str(audit/'duckdb_tmp').replace('\\','/').replace("'","''")+"'")
    # Validate ordered concatenation of all four fragments against the merged input.
    counts=[]
    def shard_frames():
        for p in parts:
            n=0
            for t in read(p):
                assert t.columns.tolist()==TIME;n+=len(t);yield t
            counts.append(n)
    shards=Cursor(shard_frames());old9=Cursor(read(first));offset=0;old_core=0
    ts=iter(read(merged))
    for a in read(full):
        assert a.columns.tolist()==CORE
        t=next(ts);assert_shared(a,t)
        tp=shards.take(len(t))
        assert_shared(t.rename(columns={'案号':'case_number','当事人':'privy'}),tp)
        assert t['裁判日期'].reset_index(drop=True).equals(tp['裁判日期']),'Judgment dates differ between shards and merged'
        r=pd.DataFrame({'rid':np.arange(offset+1,offset+len(a)+1)})
        r['full_sha']=[sha(row) for row in a.itertuples(index=False,name=None)]
        r['text_sha']=[hashlib.sha256(s.encode('utf-8')).hexdigest() if s.strip() else '' for s in a.case_text]
        r['case_key']=a.case_number.map(normal).to_numpy();r['court_key']=a.court_name.map(normal).to_numpy()
        r['case_number']=a.case_number.to_numpy();r['court_name']=a.court_name.to_numpy()
        r['is_first']=first_instance(a.case_number).to_numpy();old_core+=int(r.is_first.sum())
        core_chunk=a.loc[r.is_first.to_numpy()].reset_index(drop=True)
        if len(core_chunk):
            prior=old9.take(len(core_chunk))
            assert core_chunk.equals(prior),'Existing first-instance CSV differs from filter of directory 8; refusing remap'
        # Canonical comparison fields, without overwriting any original value.
        cols=['privy','province','city','district','specific_place','vehicle','death','lng','lat','case_dt_hour']
        for i,col in enumerate(cols):r[f'v{i}']=a[col].map(num if col in NUMERIC else normal).to_numpy()
        jd=t['裁判日期'].map(normal)
        parsed=pd.to_datetime(t['裁判日期'],format='mixed',errors='coerce')
        r['v10']=parsed.dt.strftime('%Y-%m-%d').fillna(jd).to_numpy()
        r['judgment_date']=t['裁判日期'].to_numpy()
        r['canonical_sha']=[sha(row) for row in r[[f'v{i}' for i in range(11)]].itertuples(index=False,name=None)]
        r['complete']=(r[[f'v{i}' for i in range(11)]]!='').sum(axis=1)
        c.register('batch',r)
        if offset==0:c.execute('CREATE TABLE records AS SELECT * FROM batch')
        else:c.execute('INSERT INTO records SELECT * FROM batch')
        offset+=len(a)
        if offset%100000==0:log(f'Indexed and aligned {offset:,} input rows')
    assert next(ts,None) is None,'Extra rows in merged time data'
    assert shards.take(1).empty and old9.take(1).empty,'Extra records in shards or old core'
    assert offset==sum(counts),'Time shards do not match merged output'
    log(f'Alignment verified: {offset:,} rows, shards {counts}; first-instance subset {old_core:,}')
    c.execute('CREATE TABLE exact_map AS SELECT rid,min(rid) OVER(PARTITION BY full_sha) AS e FROM records')
    c.execute('CREATE TABLE after_exact AS SELECT r.* FROM records r JOIN exact_map m USING(rid) WHERE m.e=r.rid')
    # Same case AND same nonempty court AND identical full text; canonical equality includes judgment date.
    c.execute("""CREATE TABLE canonical_map AS SELECT rid,
      CASE WHEN case_key<>'' AND court_key<>'' AND text_sha<>'' THEN
      min(rid) OVER(PARTITION BY case_key,court_key,text_sha,canonical_sha) ELSE rid END AS k
      FROM after_exact""")
    c.execute('CREATE TABLE after_canonical AS SELECT r.* FROM after_exact r JOIN canonical_map m USING(rid) WHERE m.k=r.rid')
    distincts=','.join(f"count(distinct nullif(v{i},'')) AS n{i}" for i in range(11))
    c.execute(f'''CREATE TABLE case_text_groups AS SELECT case_key,court_key,text_sha,{distincts},
      max(complete) AS most_complete,count(*) AS n FROM after_canonical GROUP BY 1,2,3''')
    no_conflict=' AND '.join(f'g.n{i}<=1' for i in range(11))
    union_complete='+'.join(f'g.n{i}' for i in range(11))
    c.execute(f"""CREATE TABLE dominance_map AS SELECT r.rid,CASE WHEN
      r.case_key<>'' AND r.court_key<>'' AND r.text_sha<>'' AND {no_conflict}
      AND g.most_complete=({union_complete}) THEN
      first_value(r.rid) OVER(PARTITION BY r.case_key,r.court_key,r.text_sha ORDER BY r.complete DESC,r.rid)
      ELSE r.rid END AS k FROM after_canonical r JOIN case_text_groups g USING(case_key,court_key,text_sha)""")
    c.execute("""CREATE TABLE decisions AS SELECT e.rid,d.k AS retained_rid,
      CASE WHEN e.rid<>e.e THEN 'exact_13_fields' WHEN k.rid<>k.k THEN 'same_case_equivalent_format'
      WHEN d.rid<>d.k THEN 'same_case_missing_only' ELSE 'retained' END AS reason
      FROM exact_map e JOIN canonical_map k ON e.e=k.rid JOIN dominance_map d ON k.k=d.rid""")
    assert c.execute('select count(*) from decisions').fetchone()[0]==offset
    assert c.execute('select count(*) from decisions d join decisions k on d.retained_rid=k.rid where k.rid<>k.retained_rid').fetchone()[0]==0
    keep=np.zeros(offset+1,dtype=bool)
    ids=c.execute('select rid from decisions where rid=retained_rid order by rid').fetchnumpy()['rid'];keep[ids]=True
    counts_by_reason=dict(c.execute('select reason,count(*) from decisions group by 1').fetchall())
    core_reasons=dict(c.execute('select d.reason,count(*) from decisions d join records r using(rid) where r.is_first group by 1').fetchall())
    c.execute('CREATE TABLE retained AS SELECT r.* FROM records r JOIN decisions d USING(rid) WHERE d.rid=d.retained_rid')
    unresolved=c.execute("select count(*) from (select case_key from retained where case_key<>'' group by 1 having count(*)>1)").fetchone()[0]
    c.execute("""COPY (SELECT d.rid AS old_row_7_8,d.retained_rid AS retained_old_row_7_8,d.reason,
      r.case_number,r.court_name,r.is_first,r.judgment_date,k.judgment_date AS retained_judgment_date
      FROM decisions d JOIN records r ON d.rid=r.rid JOIN records k ON d.retained_rid=k.rid
      WHERE d.rid<>d.retained_rid ORDER BY d.rid) TO '"""+str(audit/'removed_records.csv').replace('\\','/')+"' (HEADER,DELIMITER ',')")
    c.execute("""COPY (SELECT r.* FROM retained r JOIN (SELECT case_key FROM retained WHERE case_key<>'' GROUP BY 1 HAVING count(*)>1) g USING(case_key)
       ORDER BY case_key,rid) TO '"""+str(audit/'unresolved_case_records.csv').replace('\\','/')+"' (HEADER,DELIMITER ',')")
    c.execute("COPY (SELECT rid AS old_row_7_8,row_number() OVER(ORDER BY rid) AS new_row_7_8,is_first FROM retained ORDER BY rid) TO '"+str(audit/'retained_row_mapping.csv').replace('\\','/')+"' (HEADER,DELIMITER ',')")
    date_conflicts=c.execute("select count(*) from decisions d join records r on d.rid=r.rid join records k on d.retained_rid=k.rid where d.reason='exact_13_fields' and r.v10<>k.v10").fetchone()[0]
    log('Decisions: '+json.dumps(counts_by_reason,ensure_ascii=False)+f'; unresolved case keys: {unresolved:,}')
    destinations=[merged]+parts+[full,d8/'example.csv',first,d9/'example.csv']
    staged={p:stage/(str(i)+'_'+p.name) for i,p in enumerate(destinations)}
    sinks={p:Sink(staged[p],TIME if p.parent==d7 else CORE) for p in destinations}
    bounds=np.cumsum(counts);offset=0;core_old_row=0;core_map=[]
    ts=iter(read(merged))
    for a in read(full):
        t=next(ts);assert_shared(a,t)
        rid=np.arange(offset+1,offset+len(a)+1);mask=keep[rid]
        selected=a.loc[mask];selected_time=t.loc[mask]
        sinks[full].add(selected);sinks[merged].add(selected_time)
        for i,p in enumerate(parts):
            lo=0 if i==0 else bounds[i-1];hi=bounds[i]
            sinks[p].add(t.loc[mask & (rid>lo) & (rid<=hi)])
        fm=first_instance(a.case_number).to_numpy();selected_first=a.loc[mask & fm]
        sinks[first].add(selected_first)
        for p,df in [(d8/'example.csv',selected),(d9/'example.csv',selected_first)]:
            need=max(0,100-sinks[p].count);sinks[p].add(df.head(need))
        # Map the original core audit row IDs to the revised core CSV.
        old_core_ids=core_old_row+np.cumsum(fm)
        for old_id,all_id in zip(old_core_ids[fm & mask],rid[fm & mask]):core_map.append((int(old_id),int(all_id)))
        core_old_row+=int(fm.sum());offset+=len(a)
        if offset%100000==0:log(f'Staged {offset:,} source rows; retained {sinks[full].count:,}')
    ts.close()  # Windows requires closing the input handle before os.replace.
    outputs={str(p):s.close() for p,s in sinks.items()}
    assert sinks[full].count==int(keep.sum())==sinks[merged].count==sum(sinks[p].count for p in parts)
    assert sinks[first].count==core_reasons.get('retained',0)
    pd.DataFrame(core_map,columns=['old_core_row','old_row_7_8']).assign(new_core_row=np.arange(1,len(core_map)+1)).to_csv(audit/'core_retained_row_mapping.csv',index=False,encoding='utf-8-sig')
    # Optional existing ZIP is updated too; no ZIP is introduced if none existed.
    zip_path=d9/'ChinaTransportDatas_first_instance.zip'
    if zip_path.exists():
        import zipfile
        before[str(zip_path)]=stat(zip_path)
        zp=stage/zip_path.name
        with zipfile.ZipFile(zp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:z.write(staged[first],arcname=first.name)
        with zipfile.ZipFile(zp) as z:assert z.testzip() is None
        staged[zip_path]=zp;outputs[str(zip_path)]={'bytes':zp.stat().st_size,'member':first.name}
    for p,metadata in before.items():assert stat(Path(p))==metadata,f'Input changed during preparation: {p}'
    report={'date':datetime.now(ZoneInfo('Asia/Shanghai')).isoformat(),'input_rows_7_8':offset,'input_rows_9':old_core,
      'decisions_7_8':counts_by_reason,'decisions_9':core_reasons,'remaining_repeated_case_keys':unresolved,
      'exact13_removed_with_different_judgment_date':date_conflicts,'outputs':outputs,'status':'validated_staging',
      'rule':'Exact 13 fields; then same normalized case AND court AND identical text with equivalent fields or a single missing-only dominating row. Conflicts retained. No cross-case incident merging.',
      'judgment_date_policy':'Exact 13-field duplicates retain the first source row including its stage-7 judgment date; differing discarded judgment dates are recorded. Later same-case rules require judgment dates to agree or be missing.',
      'backups':False,'replaced':[]}
    jsonout(audit/'update_summary.json',report)
    c.execute('CHECKPOINT');c.close()
    if commit:
        for dst,tmp in staged.items():
            assert dst.resolve().is_relative_to(root/'data') and dst.parent in [d7,d8,d9]
            os.replace(tmp,dst);report['replaced'].append(str(dst));jsonout(audit/'update_summary.json',report)
        report['status']='committed';jsonout(audit/'update_summary.json',report)
        stage.rmdir()
    log('DONE '+json.dumps(report,ensure_ascii=False))
    return report,audit

def rebuild_first_instance(root):
    """Rebuild stage 9 after stage 8 regeneration, before the synchronized audit."""
    root=Path(root).resolve();d9=root/'data/9.筛选一审';d9.mkdir(exist_ok=True)
    full=root/'data/8.连接数据/ChinaTransportDatas.csv';dst=d9/'ChinaTransportDatas_first_instance.csv'
    tmp=dst.with_suffix('.csv.pending');example=d9/'example.csv';etmp=example.with_suffix('.csv.pending')
    out=Sink(tmp,CORE);ex=Sink(etmp,CORE)
    for a in read(full):
        assert a.columns.tolist()==CORE
        selected=a.loc[first_instance(a.case_number)]
        out.add(selected);ex.add(selected.head(max(0,100-ex.count)))
    stats=out.close();ex.close();os.replace(tmp,dst);os.replace(etmp,example)
    return stats

def refresh_release(root):
    """Run after exporting stage 8: stage 9 is regenerated then 7/8/9 deduplicate together."""
    rebuild_first_instance(root)
    return synchronize(root)


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'code' else Path.cwd()
if str(PROJECT_ROOT / 'code') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'code'))
import pandas as pd
import hashlib
import os
from datetime import datetime
df1 = pd.read_csv(PROJECT_ROOT / 'data/3.数据合并/裁判文书网2000-2021.csv',
                  usecols=['案号', '法院', '全文'], dtype=str, keep_default_na=False)
df2 = pd.read_csv(PROJECT_ROOT / 'data/7.匹配时间/merged_output.csv', dtype=str, keep_default_na=False)


In [ ]:
# 记录源案号对应多份文书的歧义，避免把keep='first'误认为已完成案件去重。
repeated_source = df1.loc[df1.duplicated('案号', keep=False)].copy()
repeated_source['text_sha256'] = repeated_source['全文'].map(
    lambda s: hashlib.sha256(s.encode('utf-8')).hexdigest() if s else '')
source_groups = repeated_source.groupby('案号', dropna=False).agg(
    source_rows=('案号', 'size'), court_variants=('法院', 'nunique'), text_variants=('text_sha256', 'nunique'))
source_conflicts = source_groups.loc[(source_groups.court_variants > 1) | (source_groups.text_variants > 1)]
audit_root = PROJECT_ROOT / '检查重复'
audit_root.mkdir(exist_ok=True)
source_conflicts.to_csv(audit_root / '源案号关联歧义.csv', encoding='utf-8-sig')
df1_subset = df1.drop_duplicates(subset=['案号'], keep='first')
merged_df = pd.merge(df2, df1_subset, on='案号', how='left', sort=False, validate='many_to_one')
assert len(merged_df) == len(df2), '连接造成了行数变化'
merged_df = merged_df.rename(columns={'案号':'case_number', '法院':'court_name', '当事人':'privy', '全文':'case_text'})
merged_df = merged_df.drop(columns=['裁判日期']).fillna('')
assert merged_df.columns.tolist() == CORE
print('连接记录数：', len(merged_df), '源案号歧义组：', len(source_conflicts))


In [ ]:
# 完成写入再替换8；随后同步更新7、8、9及示例文件。
output_path = PROJECT_ROOT / 'data/8.连接数据/ChinaTransportDatas.csv'
pending = output_path.with_suffix('.csv.pending')
merged_df.to_csv(pending, index=False, encoding='utf-8-sig')
os.replace(pending, output_path)
del df1, df2, df1_subset, repeated_source, merged_df
report, audit_directory = refresh_release(PROJECT_ROOT)
print('更新状态：', report['status'])
print('7/8保留：', report['decisions_7_8'].get('retained', 0))
print('9保留：', report['decisions_9'].get('retained', 0))
print('核查目录：', audit_directory)
